In [ ]:
# Cell 1: ติดตั้ง Library ที่จำเป็นและตรวจสอบ GPU บน Google Colab
%pip install -q torch torchvision opencv-python-headless scikit-learn matplotlib pillow onnx

import torch, cv2, sklearn, matplotlib
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"OpenCV: {cv2.__version__} | scikit-learn: {sklearn.__version__}")

In [ ]:
# Cell 2: เตรียม Dataset (อัปโหลด/แตกไฟล์ datasets.zip หรือใช้จาก /content/datasets) และรันเทรนโมเดลทั้งหมด
import csv
import json
import random
import time
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple, Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.decomposition import PCA
from IPython.display import display, Image as IPyImage

# --- ตั้งค่า Path บน Colab ---
DATASET_DIR = Path("/content/datasets")
WEIGHTS_OUT = Path("/content/weights/best.pt")
ANALYSIS_OUT = Path("/content/outputs/training_analysis")

# ถ้ายังไม่มีโฟลเดอร์ /content/datasets ให้ตรวจหา datasets.zip หรือให้เลือกอัปโหลด
if not DATASET_DIR.exists():
    zip_candidates = list(Path("/content").glob("*.zip"))
    if not zip_candidates:
        try:
            from google.colab import files
            print("กรุณาอัปโหลดไฟล์ datasets.zip (ที่มีโฟลเดอร์ 7 คลาส: good, shell, black, husk, broken, parchment, immature)")
            uploaded = files.upload()
            zip_candidates = [Path("/content") / name for name in uploaded.keys() if name.endswith(".zip")]
        except ImportError:
            pass
    if zip_candidates:
        with zipfile.ZipFile(zip_candidates[0], "r") as zf:
            zf.extractall("/content")
        # รองรับทั้งกรณี zip มีโฟลเดอร์ datasets/ อยู่ข้างใน หรือมี 7 คลาสอยู่ชั้นนอกสุด
        if not DATASET_DIR.exists() and (Path("/content/good").exists()):
            DATASET_DIR.mkdir(parents=True, exist_ok=True)
            for c in ["good", "shell", "black", "husk", "broken", "parchment", "immature"]:
                if (Path("/content") / c).exists():
                    (Path("/content") / c).rename(DATASET_DIR / c)

assert DATASET_DIR.exists(), f"ไม่พบโฟลเดอร์ Dataset ที่ {DATASET_DIR}"

# --- Hyperparameters ---
EPOCHS = 45
BATCH_SIZE = 28
LR = 1.2e-3
WEIGHT_DECAY = 2e-4
LABEL_SMOOTHING = 0.06
MIXUP_ALPHA = 0.20
IMG_SIZE = 224
AUG_REPEATS = 4
SEED = 42

CLASSES = ["good", "shell", "black", "husk", "broken", "parchment", "immature"]
NUM_PHYSICAL_FEATURES = 28
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_image_exif_corrected(img_path: Path) -> np.ndarray:
    pil_img = Image.open(img_path)
    pil_img = ImageOps.exif_transpose(pil_img).convert("RGB")
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

def detect_beans_in_frame(img_bgr: np.ndarray, work_scale: float = 0.25, pad_ratio: float = 0.08) -> List[Dict[str, Any]]:
    orig_h, orig_w = img_bgr.shape[:2]
    max_dim = max(orig_h, orig_w)
    if max_dim < 900:
        work_scale = 1.0
    elif max_dim < 1800:
        work_scale = 0.5
    ws_w = max(64, int(round(orig_w * work_scale)))
    ws_h = max(64, int(round(orig_h * work_scale)))
    small = cv2.resize(img_bgr, (ws_w, ws_h), interpolation=cv2.INTER_AREA)

    lab = cv2.cvtColor(small, cv2.COLOR_BGR2Lab).astype(np.float32)
    L_chan, a_chan, b_chan = cv2.split(lab)
    ksize = max(31, int(round(min(ws_w, ws_h) * 0.18)) | 1)
    bg_L = cv2.GaussianBlur(L_chan, (ksize, ksize), 0)
    bg_a = cv2.GaussianBlur(a_chan, (ksize, ksize), 0)
    bg_b = cv2.GaussianBlur(b_chan, (ksize, ksize), 0)

    dark_diff = np.clip(bg_L - L_chan, 0, 255)
    chroma_diff = np.sqrt(np.maximum(0.0, (a_chan - bg_a) ** 2 + (b_chan - bg_b) ** 2))
    fg_score_u8 = np.clip(dark_diff * 0.75 + chroma_diff * 1.85, 0, 255).astype(np.uint8)
    fg_score_u8 = cv2.GaussianBlur(fg_score_u8, (5, 5), 0)

    margin_x, margin_top, margin_bot = int(ws_w * 0.05), int(ws_h * 0.05), int(ws_h * 0.88)
    roi_mask = np.zeros_like(fg_score_u8)
    roi_mask[margin_top:margin_bot, margin_x : ws_w - margin_x] = 255
    fg_masked = cv2.bitwise_and(fg_score_u8, fg_score_u8, mask=roi_mask)

    otsu_val, _ = cv2.threshold(fg_masked[margin_top:margin_bot, margin_x : ws_w - margin_x], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, binary = cv2.threshold(fg_masked, max(28.0, float(otsu_val) * 1.08), 255, cv2.THRESH_BINARY)

    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kern, iterations=1)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kern, iterations=1)

    raw_contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_area = ws_w * ws_h
    min_bean_area, max_bean_area = img_area * 0.00045, img_area * 0.25
    valid_areas = [cv2.contourArea(c) for c in raw_contours if min_bean_area <= cv2.contourArea(c) <= max_bean_area]
    median_area = float(np.median(valid_areas)) if valid_areas else (img_area * 0.002)

    contours = []
    for cnt in raw_contours:
        area = cv2.contourArea(cnt)
        if not (min_bean_area <= area <= max_bean_area):
            continue
        if len(valid_areas) >= 3 and area > median_area * 1.55:
            mask_single = np.zeros_like(binary)
            cv2.drawContours(mask_single, [cnt], -1, 255, -1)
            dist = cv2.distanceTransform(mask_single, cv2.DIST_L2, 5)
            _, sure_fg = cv2.threshold(dist, 0.45 * dist.max(), 255, 0)
            sure_fg = sure_fg.astype(np.uint8)
            n_labels, labels = cv2.connectedComponents(sure_fg)
            if n_labels > 2:
                markers = labels + 1
                markers[cv2.subtract(mask_single, sure_fg) == 255] = 0
                cv2.watershed(cv2.cvtColor(mask_single, cv2.COLOR_GRAY2BGR), markers)
                for lbl in range(2, n_labels + 1):
                    sub_cnts, _ = cv2.findContours(np.uint8(markers == lbl) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    if sub_cnts:
                        contours.append(max(sub_cnts, key=cv2.contourArea))
                continue
        contours.append(cnt)

    beans = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_bean_area:
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        if max(w, h) / max(min(w, h), 1) > 3.5:
            continue
        x1, y1 = int(round(x / work_scale)), int(round(y / work_scale))
        bw, bh = int(round(w / work_scale)), int(round(h / work_scale))
        pad_x, pad_y = int(round(bw * pad_ratio)), int(round(bh * pad_ratio))
        px1, py1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
        px2, py2 = min(orig_w, x1 + bw + pad_x), min(orig_h, y1 + bh + pad_y)
        crop_bgr = img_bgr[py1:py2, px1:px2].copy()
        if crop_bgr.size > 0:
            beans.append({"bbox_padded": (px1, py1, px2, py2), "center": (x1 + bw / 2.0, y1 + bh / 2.0), "crop_bgr": crop_bgr, "rel_area": float((bw * bh) / max(orig_w * orig_h, 1))})
    if not beans:
        beans.append({"bbox_padded": (0, 0, orig_w, orig_h), "center": (orig_w / 2.0, orig_h / 2.0), "crop_bgr": img_bgr.copy(), "rel_area": 0.0035})
    beans.sort(key=lambda b: (int(b["center"][1] // 250), b["center"][0]))
    return beans

def segment_single_bean_crop(crop_bgr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    h, w = crop_bgr.shape[:2]
    lab = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2Lab).astype(np.float32)
    L_c, a_c, b_c = cv2.split(lab)
    bg_L = float(np.percentile(np.concatenate([L_c[0, :], L_c[-1, :], L_c[:, 0], L_c[:, -1]]), 75))
    bg_a = float(np.median(np.concatenate([a_c[0, :], a_c[-1, :], a_c[:, 0], a_c[:, -1]])))
    bg_b = float(np.median(np.concatenate([b_c[0, :], b_c[-1, :], b_c[:, 0], b_c[:, -1]])))
    score = np.clip(np.clip(bg_L - L_c, 0, 255) * 0.75 + np.sqrt((a_c - bg_a) ** 2 + (b_c - bg_b) ** 2) * 1.85, 0, 255).astype(np.uint8)
    score = cv2.GaussianBlur(score, (5, 5), 0)
    otsu_t, _ = cv2.threshold(score, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, mask = cv2.threshold(score, max(20.0, float(otsu_t) * 0.95), 255, cv2.THRESH_BINARY)
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(cv2.morphologyEx(mask, cv2.MORPH_OPEN, kern, iterations=1), cv2.MORPH_CLOSE, kern, iterations=2)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean_mask = np.zeros((h, w), dtype=np.uint8)
    if cnts:
        main_cnt = max(cnts, key=cv2.contourArea)
        cv2.drawContours(clean_mask, [main_cnt], -1, 255, thickness=cv2.FILLED)
    else:
        clean_mask[:] = 255
        main_cnt = np.array([[[0, 0]], [[w - 1, 0]], [[w - 1, h - 1]], [[0, h - 1]]], dtype=np.int32)
    return clean_mask, main_cnt

def extract_bean_physical_features(crop_bgr: np.ndarray, rel_area: float = 0.0035) -> np.ndarray:
    std_crop = cv2.resize(crop_bgr, (160, 160), interpolation=cv2.INTER_AREA)
    mask, cnt = segment_single_bean_crop(std_crop)
    h, w = std_crop.shape[:2]
    area, perimeter = float(cv2.contourArea(cnt)), float(cv2.arcLength(cnt, True))
    hull_area = float(cv2.contourArea(cv2.convexHull(cnt)))
    _, _, bw, bh = cv2.boundingRect(cnt)
    solidity = area / max(hull_area, 1.0)
    circularity = (4.0 * np.pi * area) / max(perimeter * perimeter, 1.0)
    extent = area / max(float(bw * bh), 1.0)
    if len(cnt) >= 5:
        (_, _), (ma, MA), _ = cv2.fitEllipse(cnt)
        aspect_ratio = float(max(MA, ma) / max(min(MA, ma), 1.0))
    else:
        aspect_ratio = float(max(bw, bh) / max(min(bw, bh), 1.0))
    equiv_diam = np.sqrt(4.0 * area / np.pi) if area > 0 else 1.0
    defect_depths = []
    if len(cnt) >= 5:
        hull_idx = cv2.convexHull(cnt, returnPoints=False)
        if hull_idx is not None and len(hull_idx) > 3:
            try:
                defects = cv2.convexityDefects(cnt, hull_idx)
                if defects is not None:
                    for _, _, _, d in defects.reshape(-1, 4):
                        defect_depths.append((float(d) / 256.0) / max(equiv_diam, 1.0))
            except cv2.error:
                pass
    max_defect = float(max(defect_depths)) if defect_depths else 0.0
    sum_defect = float(sum(d for d in defect_depths if d > 0.02))
    area_ratio_in_crop = area / float(h * w)

    lab = cv2.cvtColor(std_crop, cv2.COLOR_BGR2Lab).astype(np.float32)
    hsv = cv2.cvtColor(std_crop, cv2.COLOR_BGR2HSV).astype(np.float32)
    gray = cv2.cvtColor(std_crop, cv2.COLOR_BGR2GRAY).astype(np.float32)
    bean_pts = (mask > 0) if np.any(mask > 0) else np.ones((h, w), dtype=bool)
    bg_pts = mask == 0
    L_vals, a_vals, b_vals = lab[:, :, 0][bean_pts] * (100.0 / 255.0), lab[:, :, 1][bean_pts] - 128.0, lab[:, :, 2][bean_pts] - 128.0
    S_vals = hsv[:, :, 1][bean_pts] / 255.0
    if np.any(bg_pts):
        bg_L, bg_a, bg_b = float(np.median(lab[:, :, 0][bg_pts]) * (100.0 / 255.0)), float(np.median(lab[:, :, 1][bg_pts]) - 128.0), float(np.median(lab[:, :, 2][bg_pts]) - 128.0)
    else:
        bg_L, bg_a, bg_b = 85.0, 0.0, 2.0

    L_mean, L_std, L_p10, L_p90 = float(np.mean(L_vals)), float(np.std(L_vals)), float(np.percentile(L_vals, 10)), float(np.percentile(L_vals, 90))
    a_mean, a_std, b_mean, b_std = float(np.mean(a_vals)), float(np.std(a_vals)), float(np.mean(b_vals)), float(np.std(b_vals))
    S_mean, S_std = float(np.mean(S_vals)), float(np.std(S_vals))
    delta_L, delta_chroma = bg_L - L_mean, float(np.sqrt((a_mean - bg_a) ** 2 + (b_mean - bg_b) ** 2))

    dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    max_dist = float(dist_map.max()) if dist_map.max() > 0 else 1.0
    core_mask, rim_mask = (dist_map > 0.45 * max_dist), (dist_map > 0.08 * max_dist) & (dist_map <= 0.45 * max_dist)
    L_img = lab[:, :, 0] * (100.0 / 255.0)
    core_L = float(np.mean(L_img[core_mask])) if np.any(core_mask) else L_mean
    rim_L = float(np.mean(L_img[rim_mask])) if np.any(rim_mask) else L_mean
    core_minus_rim_L = core_L - rim_L
    core_std_L = float(np.std(L_img[core_mask])) if np.any(core_mask) else L_std

    inner_pts = (dist_map > 0.15 * max_dist)
    grad_mag = np.sqrt(cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3) ** 2 + cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3) ** 2)
    internal_grad_mean = float(np.mean(grad_mag[inner_pts])) if np.any(inner_pts) else 0.0
    internal_grad_p90 = float(np.percentile(grad_mag[inner_pts], 90)) if np.any(inner_pts) else 0.0

    q = (gray.astype(np.int32) * 16) // 256
    diff_h, diff_v = (q[:, 1:] - q[:, :-1]).astype(np.float32), (q[1:, :] - q[:-1, :]).astype(np.float32)
    glcm_contrast = float(0.5 * (np.mean(diff_h ** 2) + np.mean(diff_v ** 2)))
    glcm_homogeneity = float(0.5 * (np.mean(1.0 / (1.0 + diff_h ** 2)) + np.mean(1.0 / (1.0 + diff_v ** 2))))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    bh_energy = float(np.mean(blackhat[inner_pts])) if np.any(inner_pts) else 0.0
    redness_over_dark = a_mean / max(100.0 - L_mean, 5.0)

    feats = np.array([
        solidity, circularity, extent, aspect_ratio, max_defect, sum_defect, area_ratio_in_crop,
        L_mean / 100.0, L_std / 50.0, L_p10 / 100.0, L_p90 / 100.0, a_mean / 50.0, a_std / 25.0,
        b_mean / 50.0, b_std / 25.0, S_mean, S_std, delta_L / 100.0, delta_chroma / 50.0,
        core_minus_rim_L / 30.0, core_std_L / 30.0, internal_grad_mean / 100.0, internal_grad_p90 / 250.0,
        glcm_contrast / 10.0, glcm_homogeneity, bh_energy / 30.0, redness_over_dark, rel_area * 100.0
    ], dtype=np.float32)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

def preprocess_crop_to_tensor(crop_bgr: np.ndarray, img_size: int = 224) -> torch.Tensor:
    h, w = crop_bgr.shape[:2]
    scale = img_size / float(max(h, w))
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    resized = cv2.resize(crop_bgr, (nw, nh), interpolation=cv2.INTER_AREA)
    border_vals = np.concatenate([resized[0, :], resized[-1, :], resized[:, 0], resized[:, -1]], axis=0)
    pad_color = [int(x) for x in np.median(border_vals, axis=0)]
    canvas = np.full((img_size, img_size, 3), pad_color, dtype=np.uint8)
    y0, x0 = (img_size - nh) // 2, (img_size - nw) // 2
    canvas[y0 : y0 + nh, x0 : x0 + nw] = resized
    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return torch.from_numpy(((rgb - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1)).float()

def augment_bean_crop_bgr(crop_bgr: np.ndarray) -> np.ndarray:
    img = crop_bgr.copy()
    h, w = img.shape[:2]
    border_vals = np.concatenate([img[0, :], img[-1, :], img[:, 0], img[:, -1]], axis=0)
    border_color = tuple(int(x) for x in np.median(border_vals, axis=0))
    if random.random() < 0.5: img = cv2.flip(img, 1)
    if random.random() < 0.5: img = cv2.flip(img, 0)
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), random.uniform(0.0, 360.0), random.uniform(0.88, 1.12))
    M[0, 2] += random.uniform(-0.05 * w, 0.05 * w)
    M[1, 2] += random.uniform(-0.05 * h, 0.05 * h)
    img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=border_color)
    if random.random() < 0.8:
        img = np.clip(img.astype(np.float32) * random.uniform(0.90, 1.10) + random.uniform(-12.0, 12.0), 0, 255).astype(np.uint8)
    return img

class CoffeeBeanDataset(Dataset):
    def __init__(self, samples, feat_mean, feat_std, img_size=224, is_train=True, repeats=1):
        self.samples, self.feat_mean, self.feat_std, self.img_size, self.is_train = samples, feat_mean, feat_std, img_size, is_train
        self.repeats = repeats if is_train else 1
    def __len__(self):
        return len(self.samples) * self.repeats
    def __getitem__(self, idx):
        item = self.samples[idx % len(self.samples)]
        crop, phys = item["crop_bgr"], item["phys_raw"].copy()
        phys_norm = (phys - self.feat_mean) / self.feat_std
        if self.is_train:
            crop = augment_bean_crop_bgr(crop)
            phys_norm = phys_norm + np.random.normal(0.0, 0.05, size=phys_norm.shape).astype(np.float32)
        return preprocess_crop_to_tensor(crop, self.img_size), torch.from_numpy(phys_norm).float(), int(item["label"])

class HybridBeanNet(nn.Module):
    def __init__(self, num_classes=7, num_phys_features=NUM_PHYSICAL_FEATURES, pretrained=True):
        super().__init__()
        backbone = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = backbone.features
        self.phys_mlp = nn.Sequential(nn.Linear(num_phys_features, 64), nn.BatchNorm1d(64), nn.Hardswish(inplace=True), nn.Dropout(0.20), nn.Linear(64, 64), nn.BatchNorm1d(64), nn.Hardswish(inplace=True))
        self.fusion_proj = nn.Sequential(nn.Linear(576 + 64, 192), nn.BatchNorm1d(192), nn.Hardswish(inplace=True), nn.Dropout(0.30))
        self.classifier = nn.Linear(192, num_classes)
    def extract_conv_stages(self, x):
        stages = {}
        x = self.features[1](self.features[0](x))
        stages["Stage 1"] = x
        x = self.features[3](self.features[2](x))
        stages["Stage 2"] = x
        for i in range(4, 7): x = self.features[i](x)
        stages["Stage 3"] = x
        for i in range(7, 9): x = self.features[i](x)
        stages["Stage 4"] = x
        for i in range(9, 13): x = self.features[i](x)
        stages["Stage 5"] = x
        return stages
    def forward(self, img, phys, return_embedding=False):
        fmap = self.features(img)
        conv_vec = 0.7 * F.adaptive_avg_pool2d(fmap, 1).flatten(1) + 0.3 * F.adaptive_max_pool2d(fmap, 1).flatten(1)
        emb = self.fusion_proj(torch.cat([conv_vec, self.phys_mlp(phys)], dim=1))
        logits = self.classifier(emb)
        return (logits, F.normalize(emb, p=2, dim=1)) if return_embedding else (logits, emb)

@torch.no_grad()
def evaluate_hybrid(model, samples, feat_mean, feat_std, rf_clf, prototypes, img_size, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, y_true, y_cnn_pred, y_ens_pred, embs_list = 0.0, [], [], [], []
    for item in samples:
        crop, label = item["crop_bgr"], int(item["label"])
        phys_norm = (item["phys_raw"] - feat_mean) / feat_std
        imgs_t = torch.stack([preprocess_crop_to_tensor(crop, img_size), preprocess_crop_to_tensor(cv2.flip(crop, 1), img_size)], dim=0).to(device)
        phys_t = torch.from_numpy(np.stack([phys_norm, phys_norm], axis=0)).float().to(device)
        logits, embs = model(imgs_t, phys_t, return_embedding=True)
        mean_logits = torch.mean(logits, dim=0, keepdim=True)
        total_loss += float(criterion(mean_logits, torch.tensor([label], device=device)).item())
        cnn_prob = F.softmax(mean_logits, dim=1)[0].cpu().numpy()
        mean_emb = F.normalize(torch.mean(embs, dim=0, keepdim=True), p=2, dim=1)[0].cpu().numpy()
        embs_list.append(mean_emb)
        proto_prob = np.exp(np.dot(prototypes, mean_emb) * 6.0)
        proto_prob /= np.sum(proto_prob)
        rf_prob = rf_clf.predict_proba(phys_norm.reshape(1, -1))[0]
        ens_prob = 0.55 * cnn_prob + 0.20 * proto_prob + 0.25 * rf_prob
        y_true.append(label)
        y_cnn_pred.append(int(np.argmax(cnn_prob)))
        y_ens_pred.append(int(np.argmax(ens_prob)))
    return total_loss / max(len(samples), 1), accuracy_score(y_true, y_cnn_pred), accuracy_score(y_true, y_ens_pred), y_true, y_ens_pred, np.vstack(embs_list)

# --- รัน Pipeline ทั้งหมด ---
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INIT] Device: {device} | Dataset: {DATASET_DIR}")

all_samples, class_rep_samples = [], {}
for cls_idx, cls_name in enumerate(CLASSES):
    cls_dir = DATASET_DIR / cls_name
    cls_beans = []
    for img_p in sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]):
        for b in detect_beans_in_frame(load_image_exif_corrected(img_p)):
            cls_beans.append({"class_name": cls_name, "label": cls_idx, "crop_bgr": b["crop_bgr"], "phys_raw": extract_bean_physical_features(b["crop_bgr"], b["rel_area"])})
    print(f"  -> {cls_name:<10}: {len(cls_beans)} beans")
    class_rep_samples[cls_name] = cls_beans[min(3, len(cls_beans) - 1)]
    all_samples.extend(cls_beans)

train_samples, val_samples = [], []
for cls_idx in range(len(CLASSES)):
    for i, item in enumerate([s for s in all_samples if s["label"] == cls_idx]):
        (val_samples if i % 5 == 2 else train_samples).append(item)
print(f"[SPLIT] Train: {len(train_samples)} beans | Val Holdout: {len(val_samples)} beans")

X_train_phys = np.vstack([s["phys_raw"] for s in train_samples])
feat_mean, feat_std = np.mean(X_train_phys, axis=0).astype(np.float32), np.maximum(np.std(X_train_phys, axis=0), 1e-4).astype(np.float32)
rf_clf = ExtraTreesClassifier(n_estimators=250, max_depth=10, class_weight="balanced", random_state=SEED).fit((X_train_phys - feat_mean) / feat_std, [s["label"] for s in train_samples])

train_loader = DataLoader(CoffeeBeanDataset(train_samples, feat_mean, feat_std, IMG_SIZE, True, AUG_REPEATS), batch_size=BATCH_SIZE, shuffle=True)
model = HybridBeanNet(len(CLASSES), NUM_PHYSICAL_FEATURES, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.02)

history, best_val_acc, best_val_loss, best_state_dict, best_prototypes = [], -1.0, float("inf"), None, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, corr_tr, tot_tr = 0.0, 0, 0
    for imgs_t, phys_t, labels_t in train_loader:
        imgs_t, phys_t, labels_t = imgs_t.to(device), phys_t.to(device), labels_t.to(device)
        optimizer.zero_grad()
        if MIXUP_ALPHA > 0 and random.random() < 0.45:
            lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
            perm = torch.randperm(imgs_t.size(0), device=device)
            logits, _ = model(lam * imgs_t + (1 - lam) * imgs_t[perm], lam * phys_t + (1 - lam) * phys_t[perm])
            loss = lam * criterion(logits, labels_t) + (1 - lam) * criterion(logits, labels_t[perm])
        else:
            logits, _ = model(imgs_t, phys_t)
            loss = criterion(logits, labels_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        run_loss += float(loss.item()) * imgs_t.size(0)
        corr_tr += int((torch.argmax(logits, dim=1) == labels_t).sum().item())
        tot_tr += int(imgs_t.size(0))
    lr_now = float(optimizer.param_groups[0]["lr"])
    scheduler.step()

    model.eval()
    proto_mat = np.zeros((len(CLASSES), 192), dtype=np.float32)
    with torch.no_grad():
        for c_i in range(len(CLASSES)):
            c_items = [s for s in train_samples if s["label"] == c_i]
            c_imgs = torch.stack([preprocess_crop_to_tensor(s["crop_bgr"], IMG_SIZE) for s in c_items]).to(device)
            c_phys = torch.from_numpy(np.vstack([(s["phys_raw"] - feat_mean) / feat_std for s in c_items])).float().to(device)
            _, c_embs = model(c_imgs, c_phys, return_embedding=True)
            proto_mat[c_i] = F.normalize(torch.mean(c_embs, dim=0, keepdim=True), p=2, dim=1)[0].cpu().numpy()

    va_loss, va_cnn, va_ens, y_va_t, y_va_p, _ = evaluate_hybrid(model, val_samples, feat_mean, feat_std, rf_clf, proto_mat, IMG_SIZE, device)
    history.append({"epoch": epoch, "train_loss": run_loss / tot_tr, "train_acc": corr_tr / tot_tr, "val_loss": va_loss, "val_acc": va_cnn, "val_ensemble_acc": va_ens, "lr": lr_now})
    if (va_ens > best_val_acc) or (abs(va_ens - best_val_acc) < 1e-6 and va_loss < best_val_loss):
        best_val_acc, best_val_loss = va_ens, va_loss
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_prototypes = proto_mat.copy()
        print(f"Epoch {epoch:2d} | Train Loss: {run_loss/tot_tr:.4f} | Val Loss: {va_loss:.4f} | Val Hybrid Acc: {va_ens*100:.2f}%  <-- BEST")

model.load_state_dict(best_state_dict)
_, _, va_best, y_va_t, y_va_p, _ = evaluate_hybrid(model, val_samples, feat_mean, feat_std, rf_clf, best_prototypes, IMG_SIZE, device)
_, _, all_best, y_all_t, y_all_p, all_embs = evaluate_hybrid(model, all_samples, feat_mean, feat_std, rf_clf, best_prototypes, IMG_SIZE, device)
print(f"\n[DONE] Val Acc: {va_best*100:.2f}% | All 175 Beans Acc: {all_best*100:.2f}%")

# --- บันทึก best.pt ---
X_all_norm = (np.vstack([s["phys_raw"] for s in all_samples]) - feat_mean) / feat_std
rf_prod = ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight="balanced", random_state=SEED).fit(X_all_norm, [s["label"] for s in all_samples])
prod_protos = np.zeros((len(CLASSES), 192), dtype=np.float32)
model.eval()
with torch.no_grad():
    for c_i in range(len(CLASSES)):
        c_items = [s for s in all_samples if s["label"] == c_i]
        c_imgs = torch.stack([preprocess_crop_to_tensor(s["crop_bgr"], IMG_SIZE) for s in c_items]).to(device)
        c_phys = torch.from_numpy(np.vstack([(s["phys_raw"] - feat_mean) / feat_std for s in c_items])).float().to(device)
        _, c_embs = model(c_imgs, c_phys, return_embedding=True)
        prod_protos[c_i] = F.normalize(torch.mean(c_embs, dim=0, keepdim=True), p=2, dim=1)[0].cpu().numpy()

rf_trees = []
for est in rf_prod.estimators_:
    t = est.tree_
    vals = t.value[:, 0, :].astype(np.float32)
    probs = vals / np.maximum(vals.sum(axis=1, keepdims=True), 1e-8)
    rf_trees.append({"children_left": t.children_left.astype(np.int32), "children_right": t.children_right.astype(np.int32), "feature": t.feature.astype(np.int32), "threshold": t.threshold.astype(np.float32), "probs": probs})

WEIGHTS_OUT.parent.mkdir(parents=True, exist_ok=True)
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)
torch.save({"architecture": "HybridBeanNet_MobileNetV3Small_OpenCV28", "classes": CLASSES, "img_size": IMG_SIZE, "num_phys_features": NUM_PHYSICAL_FEATURES, "model_state_dict": best_state_dict, "feat_mean": feat_mean, "feat_std": feat_std, "class_prototypes": prod_protos, "rf_trees": rf_trees}, WEIGHTS_OUT)
class _ConvExt(nn.Module):
    def __init__(self, net): super().__init__(); self.features = net.features
    def forward(self, img): fmap = self.features(img); return 0.7 * F.adaptive_avg_pool2d(fmap, 1).flatten(1) + 0.3 * F.adaptive_max_pool2d(fmap, 1).flatten(1)
def _fold(l, bn): w, b, g, be, m, v = [x.detach().cpu().numpy().astype(np.float32) for x in (l.weight, l.bias, bn.weight, bn.bias, bn.running_mean, bn.running_var)]; sc = g / np.sqrt(v + float(bn.eps)); return (w * sc[:, None]).astype(np.float32), ((b - m) * sc + be).astype(np.float32)
m_cpu = model.cpu().eval()
torch.onnx.export(_ConvExt(m_cpu).eval(), torch.randn(1, 3, IMG_SIZE, IMG_SIZE), str(WEIGHTS_OUT.with_suffix(".onnx")), opset_version=13, dynamo=False, input_names=["img"], output_names=["conv_vec"])
W_p1, b_p1 = _fold(m_cpu.phys_mlp[0], m_cpu.phys_mlp[1]); W_p2, b_p2 = _fold(m_cpu.phys_mlp[4], m_cpu.phys_mlp[5]); W_fus, b_fus = _fold(m_cpu.fusion_proj[0], m_cpu.fusion_proj[1])
t_off = np.zeros(len(rf_trees) + 1, dtype=np.int32); t_off[1:] = np.cumsum([len(t["children_left"]) for t in rf_trees])
np.savez_compressed(WEIGHTS_OUT.parent / "model_meta.npz", classes=np.array(CLASSES), img_size=np.array([IMG_SIZE], dtype=np.int32), feat_mean=feat_mean, feat_std=feat_std, class_prototypes=prod_protos, W_p1=W_p1, b_p1=b_p1, W_p2=W_p2, b_p2=b_p2, W_fus=W_fus, b_fus=b_fus, W_cls=m_cpu.classifier.weight.detach().cpu().numpy().astype(np.float32), b_cls=m_cpu.classifier.bias.detach().cpu().numpy().astype(np.float32), tree_offsets=t_off, tree_cl=np.concatenate([t["children_left"] for t in rf_trees]).astype(np.int32), tree_cr=np.concatenate([t["children_right"] for t in rf_trees]).astype(np.int32), tree_feat=np.concatenate([t["feature"] for t in rf_trees]).astype(np.int32), tree_thr=np.concatenate([t["threshold"] for t in rf_trees]).astype(np.float32), tree_probs=np.vstack([t["probs"] for t in rf_trees]).astype(np.float32))
print(f"[SAVED] {WEIGHTS_OUT} + best.onnx + model_meta.npz")

# --- แสดงกราฟ Accuracy / Loss และ Feature Map ผ่าน Convolution Layers บน Colab ---
epochs_x = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=130)
axes[0].plot(epochs_x, [h["train_loss"] for h in history], label="Train Loss")
axes[0].plot(epochs_x, [h["val_loss"] for h in history], label="Val Loss")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_x, [h["train_acc"] * 100 for h in history], label="Train Acc (%)")
axes[1].plot(epochs_x, [h["val_ensemble_acc"] * 100 for h in history], label="Val Hybrid Acc (%)")
axes[1].set_title(f"Accuracy Curves (Best Val: {va_best*100:.2f}%)"); axes[1].set_ylim(0, 105); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(ANALYSIS_OUT / "training_curves_dashboard.png")
plt.show()

# --- แสดงภาพผลลัพธ์ผ่าน Convolution Stage 1..5 ของทั้ง 7 คลาส ---
fig_conv, axes_conv = plt.subplots(len(CLASSES), 6, figsize=(16, 17), dpi=120)
stage_keys = ["Stage 1", "Stage 2", "Stage 3", "Stage 4", "Stage 5"]
with torch.no_grad():
    for r, cls_name in enumerate(CLASSES):
        crop_bgr = class_rep_samples[cls_name]["crop_bgr"]
        img_t = preprocess_crop_to_tensor(crop_bgr, IMG_SIZE).unsqueeze(0).to(device)
        stages = model.extract_conv_stages(img_t)
        orig_rgb = np.clip((img_t[0].cpu().numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN) * 255.0, 0, 255).astype(np.uint8)
        axes_conv[r, 0].imshow(orig_rgb); axes_conv[r, 0].set_ylabel(cls_name.upper(), fontweight="bold")
        if r == 0: axes_conv[r, 0].set_title("Input (224x224)")
        for c_i, sk in enumerate(stage_keys):
            mean_map = np.mean(np.abs(stages[sk][0].cpu().numpy()), axis=0)
            norm_m = ((mean_map - mean_map.min()) / max(mean_map.max() - mean_map.min(), 1e-6) * 255.0).astype(np.uint8)
            hmap = cv2.cvtColor(cv2.applyColorMap(cv2.resize(norm_m, (IMG_SIZE, IMG_SIZE)), cv2.COLORMAP_VIRIDIS), cv2.COLOR_BGR2RGB)
            axes_conv[r, c_i + 1].imshow(cv2.addWeighted(orig_rgb, 0.45, hmap, 0.55, 0) if c_i == 4 else hmap)
            if r == 0: axes_conv[r, c_i + 1].set_title(sk)
        for c_i in range(6): axes_conv[r, c_i].set_xticks([]); axes_conv[r, c_i].set_yticks([])
plt.tight_layout()
fig_conv.savefig(ANALYSIS_OUT / "all_7_classes_conv_summary.png")
plt.show()